# पाठ १३ - कग्नी नॉलेज ग्राफसहित एजेन्ट मेमोरी


## सेटअप

यो नोटबुकले कसरी [**Cognee**](https://www.cognee.ai/) ज्ञान ग्राफहरू र **Microsoft Agent Framework** (MAF) प्रयोग गरी स्थायी स्मृति भएको बुद्धिमान **कोडिङ सहायक** बनाउने देखाउँछ।

Cognee ले असंरचित पाठलाई संरचित, सोधपूस गर्न सकिने ज्ञान ग्राफमा रूपान्तरण गर्दछ जुन भेक्टर एम्बेडिङहरूद्वारा समर्थन गरिएको हुन्छ — तपाईंको एजेन्टलाई धनी, सम्बन्ध-सचेत दीर्घकालीन स्मृति प्रदान गर्दछ।

### तपाईंले के सिक्नुहुनेछ
1. **ज्ञान ग्राफहरू बनाउने**: विकासकर्ता प्रोफाइलहरू र उत्तम अभ्यासहरूलाई संरचित, सोधपूस गर्न सकिने ज्ञानमा रूपान्तरण गर्ने।
2. **Cognee लाई MAF सँग एकीकृत गर्ने**: `@tool` कार्यहरू प्रयोग गरेर MAF एजेन्टलाई Cognee को ज्ञान ग्राफ सोध्न दिनुहोस्।
3. **सत्र-अबेयर संवादहरू**: एउटै सत्रमा बहुविध प्रश्नहरूमा सन्दर्भ कायम राख्नुहोस्।
4. **दीर्घकालीन स्मृति**: महत्त्वपूर्ण ज्ञान सत्रहरूमा कायम राख्नुहोस् र नयाँ संवादहरूमा तिनीहरू फिर्ता लिनुहोस्।

### आवश्यकताहरू
- पाइथन ३.९+
- स्थानीय रूपमा Redis चलिरहेको (`docker run -d -p 6379:6379 redis`) सत्र व्यवस्थापनको लागि
- LLM API कुञ्जी (जस्तै OpenAI) — `.env` मा `LLM_API_KEY` सेट गर्नुहोस्
- `.env` मा `CACHING=true` (Cognee सत्रहरूको लागि आवश्यक)
- Microsoft Foundry परियोजना जसमा लागू गरिएको च्याट मोडेल छ
- `.env` मा `AZURE_AI_PROJECT_ENDPOINT` र `AZURE_AI_MODEL_DEPLOYMENT_NAME`
- Azure CLI प्रमाणिकृत (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity "cognee[redis]==0.4.0" -q

In [ ]:
import os
from pathlib import Path
from typing import Annotated

from dotenv import load_dotenv

load_dotenv()

os.environ["LLM_API_KEY"] = os.getenv("LLM_API_KEY", "")
os.environ["CACHING"] = os.getenv("CACHING", "true")

import cognee
from cognee.modules.search.types import SearchType

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

print(f"Cognee version: {cognee.__version__}")
print(f"CACHING: {os.environ.get('CACHING')}")


In [ ]:
provider = FoundryChatClient(
    project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
    credential=AzureCliCredential(),
)

print("✅ FoundryChatClient created")


## एजेन्ट स्मृति का प्रकारहरू

यो नोटबुक मुख्य पाठ १३ को नोटबुकमा भएका तीन स्मृति प्रकारहरूलाई अन्वेषण गर्दछ, तर लामो अवधिको स्मृति ब्याकएन्डको रूपमा Cognee प्रयोग गर्दछ:

| स्मृति प्रकार | मेकानिज्म | अवधि |
|---|---|---|
| **कार्यरत** | `agent.create_session()` (MAF) | एउटै कुराकानी |
| **छोटो अवधिको** | Cognee सेसन क्याच (Redis) | एउटै सेसन |
| **लामो अवधिको** | Cognee ज्ञान ग्राफ + भेक्टरहरू | स्थायी |

### Cognee को स्मृति वास्तुकला
```
┌──────────────────────────┐
│      Raw Data            │  (developer profiles, docs, conversations)
└───────────┬──────────────┘
            │  cognee.add() + cognee.cognify()
            ▼
┌──────────────────────────────────────────┐
│  Knowledge Graph + Vector Embeddings     │
└───────────┬──────────────────────────────┘
            │  cognee.search()
            ▼
┌──────────────────┐       ┌────────────────┐
│  MAF Agent       │──────▶│  @tool funcs   │
│  (AgentSession)  │       │  wrapping       │
│                  │       │  cognee.search  │
└──────────────────┘       └────────────────┘
```


## कग्नी भण्डारण तयार गर्नुहोस्


In [ ]:
DATA_ROOT = Path('.data_storage').resolve()
SYSTEM_ROOT = Path('.cognee_system').resolve()

DATA_ROOT.mkdir(parents=True, exist_ok=True)
SYSTEM_ROOT.mkdir(parents=True, exist_ok=True)

cognee.config.data_root_directory(str(DATA_ROOT))
cognee.config.system_root_directory(str(SYSTEM_ROOT))

await cognee.prune.prune_data()
await cognee.prune.prune_system(metadata=True)
print("✅ Cognee storage configured and reset")

## भाग 1 — ज्ञान आधार निर्माण

हामी हाम्रो कोडिङ सहयोगीका लागि व्यापक ज्ञान आधार सिर्जना गर्न तीन प्रकारका डाटा ग्रहण गर्छौं:

1. **डेभलपर प्रोफाइल** — व्यक्तिगत विशेषज्ञता र प्रविधिगत पृष्ठभूमि
2. **Python सर्वोत्तम अभ्यासहरू** — व्यावहारिक निर्देशनहरूसहितको Python को जेन
3. **ऐतिहासिक संवादहरू** — डेभलपर र AI सहयोगीहरू बीचको विगतका प्रश्नोत्तर सत्रहरू


In [ ]:
developer_intro = (
    "Hi, I'm an AI/Backend engineer. "
    "I build FastAPI services with Pydantic, heavy asyncio/aiohttp pipelines, "
    "and production testing via pytest-asyncio. "
    "I've shipped low-latency APIs on AWS, Azure, and GoogleCloud."
)

python_zen_principles = """
# The Zen of Python: Practical Guide

## Key Principles With Guidance

### 1. Beautiful is better than ugly
Prefer descriptive names, clear structure, and consistent formatting.

### 2. Explicit is better than implicit
Be clear about behavior, imports, and types.

### 3. Simple is better than complex
Choose straightforward solutions first.

### 4. Flat is better than nested
Use early returns to reduce indentation.

## Modern Python Tie-ins
- Type hints reinforce explicitness
- Context managers enforce safe resource handling
- Dataclasses improve readability for data containers
"""

human_agent_conversations = """
"conversations": [
    {
      "topic": "async/await patterns",
      "user_query": "I'm building a web scraper that needs to handle thousands of URLs concurrently. What's the best way to structure this with asyncio?",
      "assistant_response": "Use asyncio with aiohttp, a semaphore to cap concurrency, TCPConnector for connection pooling, and context managers for session lifecycle."
    },
    {
      "topic": "dataclass vs pydantic",
      "user_query": "When should I use dataclasses vs Pydantic models?",
      "assistant_response": "For API input/output, prefer Pydantic: runtime validation, type coercion, JSON serialization. Integrates cleanly with FastAPI."
    },
    {
      "topic": "testing patterns",
      "user_query": "What's the best approach for pytest with async functions?",
      "assistant_response": "Use pytest-asyncio, async fixtures, and an isolated test database or mocks to reliably test async code."
    },
    {
      "topic": "error handling and logging",
      "user_query": "What's the best approach for production-ready error management?",
      "assistant_response": "Centralized error handling with custom exceptions, structured logging, and FastAPI middleware."
    }
  ]
"""

print("✅ Data sources prepared")

In [ ]:
await cognee.add(developer_intro, node_set=["developer_data"])
await cognee.add(human_agent_conversations, node_set=["developer_data"])
await cognee.add(python_zen_principles, node_set=["principles_data"])

await cognee.cognify()
print("✅ Knowledge graph built")

## ज्ञान ग्राफ दृश्य बनाउनुहोस्

Cognee ले यसले निकाल्नुभएको संस्थाहरू र सम्बन्धहरूको अन्तरक्रियात्मक HTML दृश्य प्रदान गर्न सक्छ।


In [ ]:
from cognee import visualize_graph

await visualize_graph('./cognee_graph.html')
print("📊 Graph saved to cognee_graph.html — open it in a browser to explore.")

## Memify सँग स्मरणशक्ति समृद्ध बनाउनुहोस्

`memify()` ले ज्ञान ग्राफ विश्लेषण गर्छ र बुद्धिमानी नियमहरू उत्पन्न गर्छ — जसले ढाँचा, राम्रो अभ्यासहरू, र अवधारणाहरू बीचको सम्बन्धहरू पहिचान गर्छ।


In [ ]:
await cognee.memify()
print("✅ Memory enriched with memify")

## भाग २ — Cognee उपकरणहरूसँग MAF एजेण्ट

अब हामी एक MAF एजेण्ट सिर्जना गर्दछौं जसले `@tool` कार्यहरू मार्फत Cognee को ज्ञान ग्राफ क्वेरी गर्न सक्छ। यसले एजेण्टलाई सेसनहरू मार्फत संवादात्मक सन्दर्भ कायम राख्दै ग्राफ-जागरुक सेम्यान्टिक खोजको पूर्ण शक्ति उपयोग गर्न अनुमति दिन्छ।


In [ ]:
@tool(approval_mode="never_require")
async def search_knowledge(
    query: Annotated[str, "Natural-language question to search the knowledge graph"],
) -> str:
    """Search the Cognee knowledge graph for relevant developer knowledge, best practices, and past conversations."""
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
    )
    if not results:
        return "No relevant knowledge found."
    return str(results)


@tool(approval_mode="never_require")
async def search_principles(
    query: Annotated[str, "Question about Python principles or best practices"],
) -> str:
    """Search only the Python principles subset of the knowledge graph."""
    from cognee.modules.engine.models.node_set import NodeSet
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
        node_type=NodeSet,
        node_name=["principles_data"],
    )
    if not results:
        return "No relevant principles found."
    return str(results)


print("✅ Cognee tools defined: search_knowledge, search_principles")

In [ ]:
coding_agent = provider.as_agent(
    name="CodingAssistant",
    instructions=(
        "You are an expert coding assistant with access to a knowledge graph "
        "containing developer profiles, Python best practices, and past conversations.\n\n"
        "WORKFLOW:\n"
        "1. Use search_knowledge() to find relevant information from the full knowledge graph.\n"
        "2. Use search_principles() when the question is specifically about Python best practices.\n"
        "3. Combine retrieved knowledge with your own expertise to give comprehensive answers.\n"
        "4. Reference the developer's known tech stack (FastAPI, asyncio, Pydantic) when relevant."
    ),
)

print("✅ CodingAssistant agent created")


## सत्रहरूसँग काम गर्ने स्मृति

`AgentSession` (जो `agent.create_session()` मार्फत सिर्जना गरिएको हुन्छ) सत्र भित्र काम गर्ने स्मृति प्रदान गर्दछ। एजेन्टले पहिलेका सन्देशहरूमा सन्दर्भ गर्न सक्छ र साथै Cognee को दीर्घकालीन ज्ञान ग्राफसँग प्रश्न सोध्न सक्छ।


In [ ]:
session = coding_agent.create_session()

response = await coding_agent.run(
    "How does my AsyncWebScraper implementation align with Python's design principles?",
    session=session,
)
print("🤖 Agent:", response)

In [ ]:
response = await coding_agent.run(
    "Based on what you just said, when should I pick dataclasses versus Pydantic for this work?",
    session=session,
)
print("🤖 Agent:", response)
print("\n💡 The agent combined working memory (previous answer) with Cognee's knowledge graph.")

## नयाँ सत्र — दीर्घकालीन स्मृति कायम गर्दछ

नयाँ सत्र सुरु गर्दा कार्य गर्ने स्मृति खाली हुन्छ, तर Cognee ज्ञान ग्राफ अझै उपलब्ध छ। एजेन्टले नयाँ संवादमा पनि सोही दीर्घकालीन ज्ञान फेच गर्न सक्छ।


In [ ]:
session_2 = coding_agent.create_session()

response = await coding_agent.run(
    "What logging guidance should I follow for incident reviews?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 New session, but the agent still has access to the full Cognee knowledge graph.")

In [ ]:
response = await coding_agent.run(
    "How should variables be named according to Python best practices?",
    session=session_2,
)
print("🤖 Agent:", response)

## सारांश

यस नोटबुकमा तपाईंले **MAF को कार्यशील स्मृति** (`agent.create_session()`) लाई **Cognee को दीर्घकालीन ज्ञान ग्राफ** सँग मिलाएर एक कोडिङ सहायक निर्माण गर्नुभयो।

### तपाईंले के सिक्नुभयो
1. **ज्ञान ग्राफ निर्माण**: Cognee असंरचित पाठलाई लिएर ग्राफ + भेक्टर स्मृति बनाउँछ।
2. **memify सँग ग्राफ समृद्धि**: तपाइँको अवस्थित ग्राफमा आधारित प्राप्त तथ्यहरू र धनी सम्बन्धहरू।
3. **MAF + Cognee एकीकरण**: `@tool` कार्यहरूले MAF एजेन्टलाई Cognee को ग्राफलाई स्वाभाविक रूपमा सोध्न दिन्छ।
4. **कार्यशील स्मृति + दीर्घकालीन स्मृति**: `AgentSession` (`agent.create_session()` मार्फत) सत्र सन्दर्भ प्रदान गर्छ जबकि Cognee निरन्तर ज्ञान प्रदान गर्छ।
5. **NodeSets सँग फिल्टर गरिएको खोजी**: ज्ञान ग्राफका विशेष उपसमूहहरू लक्षित गर्नुहोस् (जस्तै केवल सिद्धान्तहरू)।

### मुख्य निष्कर्षहरू
- **Cognee** कच्चा पाठलाई संरचित, सम्बन्ध-ज्ञानी स्मृतिमा परिवर्तन गर्छ — सपाट भेक्टर स्टोरभन्दा धेरै सशक्त।
- **`@tool` कार्यहरू** ले MAF एजेन्टहरू र बाह्य ज्ञान प्रणालीहरूलाई सफा रूपमा जोड्छ।
- **`AgentSession`** (`agent.create_session()` मार्फत) प्रत्येक संवाद सन्दर्भलाई दीर्घकालीन ज्ञानबाट अलग राख्छ।
- एउटै ज्ञान ग्राफले धेरै सत्रहरू र एजेन्टहरूलाई सेवा दिन्छ।

### व्यवहारिक प्रयोगहरू
- **डेभलपर कोपाइलटहरू**: कोड समीक्षा, घटना विश्लेषण, वास्तुकला सहायकहरू
- **ग्राहक-मुखीन कोपाइलटहरू**: उत्पादन कागजात, बारम्बार सोधिने प्रश्नहरू, र CRM नोटहरूमा आधारित सहायता एजेन्टहरू
- **आन्तरिक विशेषज्ञ कोपाइलटहरू**: नीति, कानूनी, वा सुरक्षा सहायकहरू जो मार्गनिर्देशनहरूमा विचार गर्दछन्
- **एकीकृत डाटा तहहरू**: संरचित र असंरचित डाटालाई एउटै खोजीयोग्य ग्राफमा संयोजन गर्नुहोस्

### आगामी कदमहरू
- Cognee मा कालक्रमीक सचेतनाका साथ प्रयोग गर्नुहोस्
- डोमेन-विशिष्ट ग्राफ गुणस्तरका लागि OWL ओन्टोलोजी परिभाषित गर्नुहोस्
- समयक्रममै पुनःप्राप्तिलाई सुधार गर्न प्रयोगकर्ता प्रतिक्रिया लूपहरू थप्नुहोस्
- एउटै Cognee स्मृति तह साझा गर्ने बहु-एजेन्ट प्रणालीहरूमा विस्तार गर्नुहोस्


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:
यो दस्तावेज़ AI अनुवाद सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) प्रयोग गरेर अनुवाद गरिएको हो। हामी सही हुन प्रयास गर्छौं, तर कृपया जानकार हुनुस् कि स्वचालित अनुवादमा त्रुटिहरू वा अशुद्धताहरू हुन सक्छन्। मूल दस्तावेज़ यसको मूल भाषामा आधिकारिक स्रोत मानिनुपर्छ। महत्वपूर्ण जानकारीका लागि व्यावसायिक मानव अनुवाद सिफारिस गरिन्छ। यस अनुवादको प्रयोगबाट उत्पन्न कुनै पनि गलत बुझाइ वा त्रुटिको लागि हामी जिम्मेवार छैनौं।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
